# m53 lab · Imaging ablation lesions and real-time MRI-guided EP

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You work out when blur hides a gap in an ablation line, look for gaps in a synthetic ring lesion around a vein,
measure agreement between image gaps and electrical reconnection with kappa, see how acute edema can close a gap
on the image, budget the frame rate of real-time MRI, filter an electrogram recorded during scanning, and read the
evidence on lesion imaging and MRI-guided ablation.

| Part | What you do | Data |
| --- | --- | --- |
| A | When blur hides a gap (eq-m53-gap) | published resolutions |
| B | Gaps in a ring lesion around a vein | our own phantom |
| C | Image gaps against electrical reconnection (kappa, m35) | our own simulation |
| D | Acute edema and apparent gaps | our own model |
| E | Real-time MRI frame budget (eq-m53-frame) | published frame rates |
| F | Electrograms during scanning | published filter chain, our own signals |
| G | Reading the evidence | published results |

Published values: LGE resolution of 1.5–2 mm routinely and about 1 mm for thin walls (Kellman and Arai 2012);
real-time frame rates of 4–8 frames/s for passive catheter visualization and up to 50/s for active tracking, and
the electrogram filter chain (Tampakis et al. 2023); coronary in-plane speeds for scale (Achenbach et al. 2000).
Lesion and gap sizes, noise, repetition times, reconnection probabilities and the electrogram and artifact
waveforms are illustrative. Each part ends with **checks**; the lab needs no downloads.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import lesion as L
from epml.ct import CORONARY_SPEED

## Part A · When blur hides a gap

A straight lesion line of width $w$ with a gap of width $g$, blurred by a Gaussian of standard deviation $\sigma$
(FWHM $\Gamma = 2\sqrt{2\ln 2}\,\sigma$), reads $B(y)\,[1 - G(x)]$: the blurred band across the line times one
minus the blurred gap along it (eq-m53-gap). The line's own signal is $\operatorname{erf}(w/(2\sqrt2\sigma))$, the
gap center's is that times $1 - \operatorname{erf}(g/(2\sqrt2\sigma))$, and with a threshold at half the contrast
the smallest visible gap is $2\sqrt2\,\sigma\,\operatorname{erfinv}(1 - 0.5/\text{peak})$. Line width (4 mm) is
illustrative.

In [ ]:
fwhms = np.linspace(0.5, 4.0, 36)
for w in (2.0, 4.0, 6.0):
    g = [L.min_visible_gap(w, L.fwhm_to_sigma(f)) for f in fwhms]
    plt.plot(fwhms, np.where(np.isfinite(g), g, np.nan), label=f"line {w:.0f} mm")
for f in (1.0, 1.5, 2.0):
    plt.axvline(f, color="gray", ls=":")
plt.xlabel("blur FWHM (mm)"); plt.ylabel("smallest visible gap (mm)"); plt.legend(); plt.show()

for f in (1.0, 1.5, 2.0):
    print(f"FWHM {f} mm, 4-mm line: smallest visible gap {L.min_visible_gap(4.0, L.fwhm_to_sigma(f)):.2f} mm")
g_15 = L.min_visible_gap(4.0, L.fwhm_to_sigma(1.5))

In [ ]:
# Checks
assert abs(L.gap_center(4.0, g_15, L.fwhm_to_sigma(1.5)) - 0.5) < 1e-9
assert L.min_visible_gap(4.0, L.fwhm_to_sigma(1.0)) < g_15 < L.min_visible_gap(4.0, L.fwhm_to_sigma(2.0))

Dotted lines mark the published resolutions: about 1 mm for thin walls, 1.5–2 mm routinely (Kellman and
Arai 2012). At these, a gap of about 0.6–1.1 mm in a 4-mm line is the smallest the threshold can show.

## Part B · Gaps in a ring lesion around a vein

A ring lesion 4 mm wide around a vein ostium of radius 12 mm has gaps of 0.6, 1.5, 2.5 and 5 mm (arc length).
We blur it, add noise, sample along the ring and call any run below half contrast a gap.

In [ ]:
gaps = [(30.0, 0.6), (110.0, 1.5), (200.0, 2.5), (300.0, 5.0)]
img, x = L.ring_lesion(gaps)
rng = np.random.default_rng(53)
fig, ax = plt.subplots(1, 3, figsize=(11, 3.6))
rows = []
for a, fwhm in zip(ax, (1.0, 2.0, 3.0)):
    b = L.blur(img, x[1] - x[0], fwhm) + 0.05 * rng.normal(size=img.shape)
    found = L.detect_gaps(*L.ring_profile(b, x))
    hit = L.matched(gaps, found)
    rows.append((fwhm, hit, len(found)))
    a.imshow(b, origin="lower", cmap="gray", vmin=0, vmax=1, extent=[x[0], x[-1], x[0], x[-1]])
    a.set_title(f"FWHM {fwhm} mm: {hit.sum()} of {len(gaps)} gaps found")
plt.tight_layout(); plt.show()
for fwhm, hit, n in rows:
    print(f"FWHM {fwhm}: found {[g for (c, g), h in zip(gaps, hit) if h]} mm gaps; {n} runs below threshold in all")

In [ ]:
# Checks
assert rows[0][1].all() or rows[0][1][1:].all()      # at 1 mm, at least every gap from 1.5 mm up is found
assert rows[2][1][-1] and not rows[2][1][0]          # at 3 mm, the 5-mm gap is found and the 0.6-mm gap is not

Noise can also add false runs below threshold; the count of runs shows it.

## Part C · Image gaps against electrical reconnection

An image gap and an electrical reconnection are different findings. Here each simulated vein has at most one true
gap; gaps at least 1 mm wide usually conduct, and some veins reconnect for other reasons. All probabilities are
illustrative. Agreement between "the image shows a gap" and "the vein reconnected" is measured with Cohen's kappa
(m35). In a clinical study, the extent of image gaps correlated with reconnection but their location agreed only
fairly (kappa 0.37) (Al Jefairi et al. 2019).

In [ ]:
w, rec = L.simulate_veins(np.random.default_rng(7))
print(f"veins with a true gap {np.mean(w > 0):.0%}; reconnected {rec.mean():.0%}")
res = []
for fwhm in (0.5, 1.0, 1.5, 2.0, 3.0, 4.0):
    det = L.image_detects(w, 4.0, fwhm)
    res.append((fwhm, det.mean(), L.kappa(det, rec)))
    print(f"FWHM {fwhm}: image shows a gap in {det.mean():.0%} of veins; kappa with reconnection {res[-1][2]:.2f}")
perfect = L.kappa(w > 0, rec)
print(f"even a perfect image of every true gap: kappa {perfect:.2f}")

In [ ]:
# Checks
assert perfect < 0.9                     # reconnection is not the same thing as an anatomical gap
assert all(-1 <= k <= 1 for _, _, k in res)

Even a perfect image of the tissue does not predict reconnection perfectly in this model, because thin gaps
need not conduct and veins can reconnect for other reasons. Kappa then measures the agreement between two
different questions, not the scanner alone.

## Part D · Acute edema and apparent gaps

Right after ablation, edema-sensitive imaging shows a region larger than the irreversible lesion, which shrinks as
edema resolves (Tampakis et al. 2023). If the edema extends a margin $e$ beyond each lesion, a gap $g$ looks like
$\max(0, g - 2e)$. The margins here are illustrative.

In [ ]:
gs = np.linspace(0, 6, 61)
for e in (0.5, 1.0, 2.0):
    plt.plot(gs, [L.apparent_gap(g, e) for g in gs], label=f"edema margin {e} mm")
plt.plot(gs, gs, "k--", lw=1, label="no edema")
plt.xlabel("true gap (mm)"); plt.ylabel("apparent gap (mm)"); plt.legend(); plt.show()
hidden = [g for g in (1.0, 2.0, 3.0, 4.0) if L.apparent_gap(g, 1.0) < L.min_visible_gap(4.0, L.fwhm_to_sigma(1.5))]
print("with a 1-mm margin and 1.5-mm blur, these gaps look closed:", hidden)

In [ ]:
# Checks
assert L.apparent_gap(2.0, 1.0) == 0.0 and L.apparent_gap(5.0, 1.0) == 3.0

## Part E · Real-time MRI frame budget

A real-time image is acquired line by line, one repetition time TR apart: a frame takes
$T = N_{\text{lines}}\,\mathrm{TR}/R$ with acceleration factor $R$ (eq-m53-frame). Passive catheter visualization
has been reported at 4–8 frames/s and active tracking of tip coils at up to 50/s (Tampakis et al. 2023). The TR
values here are illustrative. For scale, coronary arteries move 22–70 mm/s in plane (Achenbach et al. 2000).

In [ ]:
ns = np.arange(16, 257, 8)
for tr in (2.5, 3.0, 4.0):
    plt.plot(ns, [L.fps(n, tr) for n in ns], label=f"TR {tr} ms")
plt.axhspan(*L.FPS_PASSIVE, color="C2", alpha=0.15, label="4-8 /s (passive)")
plt.axhline(L.FPS_ACTIVE, color="C4", ls="--", label="50 /s (active tracking)")
plt.ylim(0, 75); plt.xlabel("lines per frame"); plt.ylabel("frames per second"); plt.legend(fontsize=8); plt.show()

ft = L.frame_time_ms(64, 3.0)
print(f"64 lines at 3 ms: {ft:.0f} ms per frame, {L.fps(64, 3.0):.1f} frames/s")
for name in ("left anterior descending", "all three arteries", "right coronary"):
    print(f"  at {CORONARY_SPEED[name]} mm/s ({name}): {L.motion_per_frame_mm(CORONARY_SPEED[name], ft):.1f} mm per frame")
print(f"active tracking at 50/s: {L.motion_per_frame_mm(CORONARY_SPEED['all three arteries'], 1e3 / L.FPS_ACTIVE):.1f} mm per update")

In [ ]:
# Checks
assert L.FPS_PASSIVE[0] <= L.fps(64, 3.0) <= L.FPS_PASSIVE[1]
assert L.fps(64, 3.0, 2.0) == 2 * L.fps(64, 3.0)

Imaging a whole plane costs tens of lines per frame; locating a few tip coils costs far fewer, which is why
active tracking can update so much faster.

## Part F · Electrograms during scanning

Scanning disturbs electrograms through magnetohydrodynamic voltages, RF pulses and gradient switching; one setup
used a 300 Hz low-pass, a 30 Hz high-pass and a 60 Hz notch, and found the best signals with a 120 Hz low-pass,
at the cost of lower amplitude (Tampakis et al. 2023). The electrogram and the interference below are illustrative
waveforms, not recordings.

In [ ]:
fs = 2000.0
t, s, onsets = L.synthetic_egm(fs=fs)
art = L.gradient_artifact(t)
raw = s + art
out = {}
for lp in (300.0, 120.0):
    y = L.filter_chain(raw, fs, low_pass=lp)
    clean = L.filter_chain(s, fs, low_pass=lp)
    out[lp] = (y, float(np.sqrt(np.mean((y - clean) ** 2))), L.peak_to_peak_at(clean, t, onsets))
seg = (t > 0.05) & (t < 0.65)
fig, ax = plt.subplots(3, 1, figsize=(8, 5), sharex=True)
ax[0].plot(t[seg], raw[seg], lw=0.6); ax[0].set_title("electrogram + scanner interference (illustrative)")
for a, lp in zip(ax[1:], (300.0, 120.0)):
    a.plot(t[seg], out[lp][0][seg], lw=0.8); a.set_title(f"after the chain, {lp:.0f} Hz low-pass")
ax[2].set_xlabel("time (s)"); [a.set_ylabel("mV") for a in ax]
plt.tight_layout(); plt.show()
for lp, (_, rms, p2p) in out.items():
    print(f"{lp:.0f} Hz: residual interference {rms:.3f} mV RMS; electrogram {p2p:.2f} mV peak to peak")
print(f"interference before filtering: {np.sqrt(np.mean(art ** 2)):.3f} mV RMS")

In [ ]:
# Checks
assert out[120.0][1] < out[300.0][1] and out[120.0][2] < out[300.0][2]

Lower cut-offs remove more interference and more of the electrogram; low-amplitude signals such as late or
double potentials are the first to suffer.

## Part G · Reading the evidence

| Study | Question | Design | Finding |
| --- | --- | --- | --- |
| Al Jefairi et al. 2019 | Do scar gaps on MRI three months after PVI match reconnected veins? | Prospective cohort, 51 patients | Gaps in 76% of patients; extent correlated with reconnection; location agreement fair (kappa 0.37) |
| Akoum et al. 2024 | Did lesions become scar in a trial of fibrosis-guided ablation? | Secondary analysis of a trial | Not uniformly; large differences between centers |
| Hopman et al. 2026 | Real-time MRI-guided flutter ablation in early clinical use | Retrospective single-center series, 35 patients | Acute block 32 of 35; technical challenges in 19 |
| Jahnke et al. 2024 | Ready-to-use meshes from MRI for EP | Series, 42 patients | Whole-heart scan about 98 s; meshes in about 16 s |
| Tampakis et al. 2023 | State of real-time MRI-guided ablation | Narrative review | Human use limited to typical flutter; acute success 93–100% in recent larger studies |

In [ ]:
designs = {"Al Jefairi 2019": "cohort", "Akoum 2024": "trial analysis", "Hopman 2026": "series",
           "Jahnke 2024": "series", "Tampakis 2023": "review"}
randomized_imri = [k for k, v in designs.items() if v == "randomized trial"]
print("Randomized trials of real-time MRI-guided ablation in this list:", len(randomized_imri))
print("Acute block in the 2026 series:", f"{32 / 35:.0%}")

In [ ]:
# Checks
assert len(randomized_imri) == 0

The evidence on real-time MRI-guided ablation so far is feasibility in typical flutter, from series and reviews.

## Where to go next

- In part A, how much narrower must the blur be to see a 0.5-mm gap in a 2-mm line?
- In part B, raise the noise to 0.15. How many false gaps appear, and how would you suppress them?
- In part C, make every gap conduct (`p_conduct=1`, `p_other=0`). What limits kappa now?
- In part E, what lines per frame and TR would give 20 frames/s without acceleration?